# Direction X — Limitation-driven mutation-aware and bias-robust adaptive fusion

Notebook này phát triển tiếp từ Direction U/V, nhưng không chỉ “thêm model”. Mục tiêu là **đánh vào hạn chế của các hướng trước và của các bài AMR prediction dùng annotation/gene presence-absence**:

1. **Mutation-driven resistance gap**: CIP/NAL nhóm quinolone/fluoroquinolone thường liên quan `gyrA/gyrB/parC/parE`, `qnr`, efflux/regulatory genes. Nếu chỉ dùng annotation presence/absence thì khó bắt mutation signal.
2. **Sampling / metadata bias**: random split có thể quá dễ nếu train/test cùng serovar, source, country, year.
3. **Single-marker over-reliance**: model điểm cao nhưng phụ thuộc quá mạnh vào 1 marker thì khó generalize.
4. **Performance target**: cố gắng nâng NAL/STR và giữ CIP/TET/GEN ổn định.

Output chính:

- `direction_X_best_summary.csv`
- `direction_X_all_metrics.csv`
- `direction_X_bias_split_audit.csv`
- `direction_X_feature_dropout_stress_test.csv`
- `direction_X_top_feature_summary_annotated.csv`
- `AUTO_CONCLUSION_DIRECTION_X.md`
- `fig_direction_X_*.png`

Chạy tốt nhất sau Direction U/V trong cùng Colab để dùng lại `direction_U_*_model_rows.csv` và `direction_U_*_tokens_by_genome.json`. Nếu không có, notebook sẽ tự clone metadata và fetch BV-BRC genome features.


In [ ]:
# =========================
# 0. Optional Colab mount
# =========================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Không chạy trên Colab hoặc không cần mount Drive:', repr(e)[:120])


In [ ]:
# =========================
# 1. Imports + global config
# =========================
import os
import re
import json
import time
import math
import shutil
import warnings
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from scipy import sparse

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# Project paths
PROJECT_NAME = 'salmonella_direction_X_limitation_driven'
BASE_DIR = Path('/content') / PROJECT_NAME if Path('/content').exists() else Path.cwd() / PROJECT_NAME
OUTPUT_DIR = BASE_DIR / 'outputs'
CACHE_DIR = BASE_DIR / 'cache_bvbrc_features'
REPO_DIR = BASE_DIR / 'AMRMetadataReview_2021'
for d in [BASE_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Search previous outputs from U/V if available.
SEARCH_ROOTS = [
    Path.cwd(),
    Path('/content') if Path('/content').exists() else None,
    Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else None,
    Path('/mnt/data') if Path('/mnt/data').exists() else None,
]
SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p is not None and p.exists()]

# Main drugs from external validation.
TARGET_DRUGS = [
    {'drug_code': 'TET', 'contains': 'tetracycline', 'drug_class': 'tetracycline'},
    {'drug_code': 'CIP', 'contains': 'ciprofloxacin', 'drug_class': 'fluoroquinolone'},
    {'drug_code': 'NAL', 'contains': 'nalidixic acid', 'drug_class': 'quinolone'},
    {'drug_code': 'GEN', 'contains': 'gentamicin', 'drug_class': 'aminoglycoside'},
    {'drug_code': 'STR', 'contains': 'streptomycin', 'drug_class': 'aminoglycoside'},
]

# Quick mode: tăng các số này nếu muốn chạy kỹ hơn.
N_PER_CLASS = 250       # 200 nhanh, 300-500 kỹ hơn nếu fetch đủ dữ liệu
N_REPEATS = 5           # 3 nhanh, 10 tốt hơn để báo cáo
TEST_SIZE = 0.20
VAL_SIZE = 0.25
RANDOM_SEED = 42

# Feature selection sizes.
K_FEATURES_LIST = [100, 200, 500, 1000]

# Model list. Dropout ensemble là điểm mới của Direction X.
MODEL_NAMES = ['lr_balanced', 'dropout_lr']
if HAS_XGB:
    MODEL_NAMES += ['xgb_weighted', 'dropout_xgb']
else:
    MODEL_NAMES += ['rf_balanced']

# Dropout ensemble parameters.
DROPOUT_ENSEMBLE_SIZE = 12
DROPOUT_RATE = 0.18

# Bias-aware validation.
RUN_BIAS_AWARE_SPLITS = True
BIAS_GROUP_COLUMNS = ['serovar', 'country', 'host', 'isolation_source', 'collection_year']
MAX_SPLITS_PER_DRUG = 4  # để tránh quá nặng: stratified + tối đa vài group splits hợp lệ

# Negative controls and stress tests.
RUN_NEGATIVE_CONTROL = True
RUN_FEATURE_DROPOUT_STRESS_TEST = True

# Optional AMRFinderPlus/CARD/ResFinder table if you have it.
USE_OPTIONAL_AMR_TABLE = False
OPTIONAL_AMR_TABLE_PATH = BASE_DIR / 'optional_amrfinder_card_resfinder.tsv'

# Optional sequence-based mutation extraction. OFF by default because BV-BRC annotation API usually may not include sequences.
# If you have gene FASTA files, put them in OPTIONAL_MUTATION_FASTA_DIR.
USE_OPTIONAL_LOCAL_MUTATION_FASTA = False
OPTIONAL_MUTATION_FASTA_DIR = BASE_DIR / 'optional_mutation_fastas'

# BV-BRC API settings.
API_BASE = 'https://www.bv-brc.org/api'
FEATURE_FIELDS = [
    'genome_id', 'patric_id', 'feature_id', 'feature_type',
    'gene', 'product', 'pgfam_id', 'plfam_id'
]
API_SLEEP = 0.10
API_LIMIT_PER_QUERY = 50000

print('HAS_XGB:', HAS_XGB)
print('BASE_DIR:', BASE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SEARCH_ROOTS:', SEARCH_ROOTS)
print('TARGET_DRUGS:', [d['drug_code'] for d in TARGET_DRUGS])
print('N_PER_CLASS:', N_PER_CLASS, '| N_REPEATS:', N_REPEATS)
print('MODEL_NAMES:', MODEL_NAMES)


## 1. Helper: tìm output cũ từ Direction U/V

Direction X ưu tiên reuse dữ liệu đã fetch ở U/V. Cần nhất là:

- `direction_U_<DRUG>_model_rows.csv`
- `direction_U_<DRUG>_tokens_by_genome.json`

Nếu không có, notebook tự fetch lại từ BV-BRC.


In [ ]:
# =========================
# 2. File discovery helpers
# =========================
def find_file(filename, roots=SEARCH_ROOTS, max_depth=6):
    for root in roots:
        root = Path(root)
        direct = root / filename
        if direct.exists():
            return direct
        try:
            # Use rglob but avoid too deep folder scans on huge Drive.
            for p in root.rglob(filename):
                if len(p.relative_to(root).parts) <= max_depth:
                    return p
        except Exception:
            pass
    return None


def find_direction_u_paths(drug_code):
    rows = find_file(f'direction_U_{drug_code}_model_rows.csv')
    toks = find_file(f'direction_U_{drug_code}_tokens_by_genome.json')
    return rows, toks


def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

print('Test file discovery for TET:')
print(find_direction_u_paths('TET'))


## 2. Load BV-BRC AMR metadata nếu cần

Nếu không tìm được Direction U outputs, phần này clone `BV-BRC/AMRMetadataReview_2021` và tạo bảng Salmonella AMR dạng long table.


In [ ]:
# =========================
# 3. AMR metadata helpers
# =========================
def ensure_metadata_repo():
    # Nếu Direction U đã clone repo ở /content, dùng lại.
    for candidate in [
        Path('/content/salmonella_direction_U_external_multidrug/AMRMetadataReview_2021'),
        Path('/content/salmonella_direction_V_robust_external_validation/AMRMetadataReview_2021'),
        REPO_DIR,
    ]:
        if candidate.exists() and (candidate / 'tabular').exists():
            print('Using existing metadata repo:', candidate)
            return candidate
    print('Cloning BV-BRC AMR metadata repo...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/BV-BRC/AMRMetadataReview_2021.git', str(REPO_DIR)],
        check=True
    )
    return REPO_DIR


def norm_col(c):
    return re.sub(r'[^a-z0-9]+', '_', str(c).lower()).strip('_')


def find_col(df, keywords):
    for c in df.columns:
        nc = norm_col(c)
        if all(k in nc for k in keywords):
            return c
    return None


def find_col_any(df, list_of_keyword_lists):
    for kws in list_of_keyword_lists:
        c = find_col(df, kws)
        if c is not None:
            return c
    return None


def parse_sir_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ['r', 'resistant', 'resistance', 'non-susceptible', 'nonsusceptible']:
        return 'R'
    if s in ['s', 'susceptible', 'sensitive', 'susceptibility']:
        return 'S'
    if s in ['i', 'intermediate']:
        return 'I'
    if 'resistant' in s and 'suscept' not in s:
        return 'R'
    if 'suscept' in s or 'sensitive' in s:
        return 'S'
    if 'intermediate' in s:
        return 'I'
    return np.nan


def normalize_antibiotic_name(x):
    s = str(x).strip()
    s = re.sub(r':.*$', '', s)
    s = s.replace('_', ' ').replace('-', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

ANTIBIOTIC_SYNONYMS = {
    'tetracycline': 'TET',
    'ciprofloxacin': 'CIP',
    'streptomycin': 'STR',
    'gentamicin': 'GEN',
    'nalidixic acid': 'NAL',
    'kanamycin': 'KAN',
    'ampicillin': 'AMP',
    'chloramphenicol': 'CHL',
}


def antibiotic_code(name):
    s = normalize_antibiotic_name(name).lower()
    if s in ANTIBIOTIC_SYNONYMS:
        return ANTIBIOTIC_SYNONYMS[s]
    for key, code in ANTIBIOTIC_SYNONYMS.items():
        if key in s:
            return code
    return re.sub(r'[^A-Za-z]', '', s).upper()[:3]


def infer_columns(df):
    return {
        'genome_id': find_col_any(df, [['genome', 'id'], ['genome_id'], ['genome']]),
        'genome_name': find_col_any(df, [['genome', 'name'], ['organism', 'name'], ['organism'], ['species'], ['taxon']]),
        'antibiotic': find_col_any(df, [['antibiotic'], ['drug'], ['antimicrobial']]),
        'phenotype': find_col_any(df, [['resistant', 'phenotype'], ['phenotype'], ['sir'], ['resistance']]),
        'serovar': find_col_any(df, [['serovar'], ['serotype']]),
        'isolation_source': find_col_any(df, [['isolation', 'source'], ['source']]),
        'collection_date': find_col_any(df, [['collection', 'date'], ['date']]),
        'country': find_col_any(df, [['country']]),
        'host': find_col_any(df, [['host']]),
    }


def read_any_table(path):
    path = Path(path)
    df = pd.read_csv(path, sep='\t', low_memory=False)
    if df.shape[1] == 1:
        df = pd.read_csv(path, low_memory=False)
    return df


def build_salmonella_amr_long_table():
    repo = ensure_metadata_repo()
    tabular_dir = repo / 'tabular'
    files = sorted([p for p in tabular_dir.glob('*') if p.is_file()])
    long_tables = []
    for p in files:
        try:
            df = read_any_table(p)
            cols = infer_columns(df)
            if cols['genome_id'] and cols['antibiotic'] and cols['phenotype']:
                tmp = pd.DataFrame()
                tmp['source_file'] = p.name
                tmp['genome_id'] = df[cols['genome_id']].astype(str)
                tmp['genome_name'] = df[cols['genome_name']].astype(str) if cols['genome_name'] else ''
                tmp['antibiotic'] = df[cols['antibiotic']].map(normalize_antibiotic_name)
                tmp['drug_code'] = tmp['antibiotic'].map(antibiotic_code)
                tmp['phenotype'] = df[cols['phenotype']].map(parse_sir_value)
                for out_col in ['serovar', 'isolation_source', 'collection_date', 'country', 'host']:
                    tmp[out_col] = df[cols[out_col]].astype(str) if cols.get(out_col) else ''
                long_tables.append(tmp)
        except Exception as e:
            print('Skip:', p.name, repr(e)[:120])
    if not long_tables:
        raise RuntimeError('Không đọc được bảng AMR nào từ repo/tabular.')
    amr = pd.concat(long_tables, ignore_index=True)
    amr = amr.dropna(subset=['genome_id', 'antibiotic', 'phenotype'])
    amr = amr[amr['phenotype'].isin(['S', 'R'])].copy()
    salm = amr[amr['genome_name'].str.contains('salmonella', case=False, na=False)].copy()
    if len(salm) == 0:
        salm = amr[amr['source_file'].str.contains('salmonella', case=False, na=False)].copy()

    def resolve_duplicate(g):
        vc = g['phenotype'].value_counts()
        keep = vc.index[0]
        row = g.iloc[0].copy()
        row['phenotype'] = keep
        row['n_duplicate_rows'] = len(g)
        row['has_conflict'] = int(g['phenotype'].nunique() > 1)
        return row

    salm = (
        salm.groupby(['genome_id', 'drug_code'], as_index=False, group_keys=False)
            .apply(resolve_duplicate)
            .reset_index(drop=True)
    )
    # Parse collection year if possible.
    salm['collection_year'] = salm['collection_date'].astype(str).str.extract(r'(19\d{2}|20\d{2})')[0]
    return salm

salm_cache_path = OUTPUT_DIR / 'direction_X_salmonella_external_long_table.csv'
if salm_cache_path.exists():
    salm = pd.read_csv(salm_cache_path)
    print('Loaded cached Salmonella long table:', salm.shape)
else:
    # Chỉ build khi thật sự cần. Nếu U outputs đủ, cell này vẫn có thể dùng để audit metadata.
    try:
        salm = build_salmonella_amr_long_table()
        salm.to_csv(salm_cache_path, index=False)
        print('Built Salmonella long table:', salm.shape)
    except Exception as e:
        salm = pd.DataFrame()
        print('Metadata loading failed. Sẽ chỉ dùng được nếu có Direction U outputs:', repr(e)[:300])

if not salm.empty:
    display(salm['drug_code'].value_counts().head(20).to_frame('n_rows'))


## 3. Fetch / load genome annotation tokens

Feature token gồm `pgfam`, `plfam`, `gene`, `product`. Đây là base annotation. Direction X sẽ thêm mechanism-proxy và mutation-proxy token lên trên base này.


In [ ]:
# =========================
# 4. Genome features + tokens
# =========================
def clean_feature_token(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s == '' or s in ['nan', 'none', 'hypothetical protein']:
        return None
    s = re.sub(r'[^a-z0-9_\-\.\(\)\+]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    if len(s) < 2:
        return None
    return s[:180]


def features_to_tokens(df):
    tokens = []
    for _, r in df.iterrows():
        for col, prefix in [('pgfam_id','pgfam'), ('plfam_id','plfam'), ('gene','gene'), ('product','prod')]:
            if col in df.columns:
                tok = clean_feature_token(r.get(col))
                if tok:
                    tokens.append(f'{prefix}:{tok}')
    return sorted(set(tokens))


def fetch_features_for_genome_eq(genome_id):
    select = ','.join(FEATURE_FIELDS)
    url = f'{API_BASE}/genome_feature/?eq(genome_id,{genome_id})&select({select})&limit({API_LIMIT_PER_QUERY})'
    headers = {'Accept': 'application/json'}
    r = requests.get(url, headers=headers, timeout=90)
    if r.status_code != 200:
        raise RuntimeError(f'HTTP {r.status_code}: {r.text[:300]}')
    return r.json()


def fetch_features_for_genome(genome_id):
    safe_id = str(genome_id).replace('.', '_').replace('/', '_')
    cache_file = CACHE_DIR / f'features_{safe_id}.json'
    if cache_file.exists():
        return load_json(cache_file)
    data = fetch_features_for_genome_eq(genome_id)
    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(data, f)
    time.sleep(API_SLEEP)
    return data


def genome_features_to_token_list(genome_id):
    try:
        data = fetch_features_for_genome(genome_id)
        if isinstance(data, dict) and 'response' in data and 'docs' in data['response']:
            data = data['response']['docs']
        if not isinstance(data, list) or len(data) == 0:
            return []
        df = pd.DataFrame(data)
        return features_to_tokens(df)
    except Exception as e:
        print('Fetch failed:', genome_id, repr(e)[:180])
        return []


def fetch_tokens_for_genomes(genome_ids, progress_every=25):
    tokens_by_genome = {}
    failed = []
    genome_ids = list(dict.fromkeys([str(g) for g in genome_ids]))
    for i, gid in enumerate(genome_ids, start=1):
        if i == 1 or i % progress_every == 0:
            print(f'Fetching {i}/{len(genome_ids)}...')
        toks = genome_features_to_token_list(gid)
        tokens_by_genome[gid] = toks
        if len(toks) == 0:
            failed.append(gid)
    return tokens_by_genome, failed


def load_or_build_drug_dataset(drug_info):
    drug_code = drug_info['drug_code']
    contains = drug_info['contains']
    # 1) Prefer Direction U cached outputs.
    rows_path, toks_path = find_direction_u_paths(drug_code)
    if rows_path is not None and toks_path is not None:
        model_df = pd.read_csv(rows_path)
        tokens_by_genome = load_json(toks_path)
        print(f'Reuse Direction U data for {drug_code}: rows={model_df.shape}, tokens={len(tokens_by_genome)}')
        if 'y' not in model_df.columns and 'phenotype' in model_df.columns:
            model_df['y'] = (model_df['phenotype'] == 'R').astype(int)
        return model_df, tokens_by_genome, 'reuse_direction_U'

    if salm.empty:
        raise RuntimeError(f'Không có Direction U output cho {drug_code} và metadata cũng không load được.')

    target = salm[(salm['drug_code'] == drug_code) | (salm['antibiotic'].str.contains(contains, case=False, na=False))].copy()
    target = target[target['phenotype'].isin(['S', 'R'])].copy()
    n_s = int((target['phenotype'] == 'S').sum())
    n_r = int((target['phenotype'] == 'R').sum())
    n_each = min(N_PER_CLASS, n_s, n_r)
    print(f'{drug_code} raw target:', target.shape, {'S': n_s, 'R': n_r}, 'n_each:', n_each)
    if n_each < 50:
        raise RuntimeError(f'{drug_code}: không đủ S/R sau lọc. S={n_s}, R={n_r}')

    target_balanced = pd.concat([
        target[target['phenotype'] == 'S'].sample(n_each, random_state=RANDOM_SEED),
        target[target['phenotype'] == 'R'].sample(n_each, random_state=RANDOM_SEED),
    ], axis=0).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    target_balanced['y'] = (target_balanced['phenotype'] == 'R').astype(int)

    genome_ids = target_balanced['genome_id'].astype(str).tolist()
    tokens_by_genome, failed = fetch_tokens_for_genomes(genome_ids)
    valid_rows = []
    for _, row in target_balanced.iterrows():
        gid = str(row['genome_id'])
        if len(tokens_by_genome.get(gid, [])) > 0:
            valid_rows.append(row.to_dict())
    model_df = pd.DataFrame(valid_rows)
    model_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_model_rows.csv', index=False)
    with open(OUTPUT_DIR / f'direction_X_{drug_code}_tokens_by_genome.json', 'w', encoding='utf-8') as f:
        json.dump(tokens_by_genome, f)
    return model_df, tokens_by_genome, 'built_from_bvbrc'


## 4. Direction X feature engineering

Có 4 feature set chính:

1. `base_annotation`: giống Direction U/V.
2. `mechanism_proxy`: thêm token theo cơ chế thuốc, ví dụ `tet`, `aac`, `aph`, `qnr`, `gyr`, `parC`, efflux.
3. `mutation_proxy`: tập trung CIP/NAL, thêm proxy cho quinolone mutation-related genes và topoisomerase/efflux regulators.
4. `mechanism_plus_optional_amr`: nếu upload AMRFinderPlus/CARD/ResFinder table thì merge thêm token từ bảng đó.

Lưu ý: `mutation_proxy` ở đây là **mutation-aware proxy** từ annotation/gene/product. Nếu bạn có FASTA sequence của `gyrA/gyrB/parC/parE`, bật cell optional bên dưới để tạo mutation signature thật.


In [ ]:
# =========================
# 5. Mechanism-aware and mutation-proxy tokens
# =========================
MECHANISM_KEYWORDS = {
    'TET': [
        'tet', 'tetracycline', 'mfs', 'efflux', 'ribosomal_protection',
        'tetra', 'transposase', 'integrase', 'repressor'
    ],
    'GEN': [
        'aac', 'aph', 'ant', 'aad', 'aminoglycoside', 'acetyltransferase',
        'phosphotransferase', 'nucleotidyltransferase', 'integron', 'inti1', 'efflux'
    ],
    'STR': [
        'str', 'stra', 'strb', 'aad', 'ant', 'aph', 'aminoglycoside',
        'phosphotransferase', 'nucleotidyltransferase', 'integron', 'inti1'
    ],
    'CIP': [
        'ciprofloxacin', 'fluoroquinolone', 'quinolone', 'qnr', 'gyr', 'gyra', 'gyrb',
        'parc', 'pare', 'topoisomerase', 'dna_gyrase', 'acr', 'acra', 'acrb',
        'mar', 'marr', 'rama', 'soxs', 'oqx', 'qep', 'efflux'
    ],
    'NAL': [
        'nalidixic', 'quinolone', 'qnr', 'gyr', 'gyra', 'gyrb', 'parc', 'pare',
        'topoisomerase', 'dna_gyrase', 'acr', 'acra', 'acrb', 'mar', 'marr',
        'rama', 'soxs', 'oqx', 'qep', 'efflux'
    ],
}

QUINOLONE_MUTATION_PROXY_TERMS = [
    'gyra', 'gyrb', 'parc', 'pare', 'dna_gyrase', 'topoisomerase',
    'qnr', 'acra', 'acrb', 'acr', 'mar', 'marr', 'soxs', 'rama', 'oqx', 'qep', 'efflux'
]


def normalize_token_text(tok):
    return str(tok).lower().replace('-', '_').replace(' ', '_')


def add_mechanism_proxy_tokens(tokens, drug_code):
    tokens = list(tokens)
    text_tokens = [normalize_token_text(t) for t in tokens]
    joined = ' '.join(text_tokens)
    new = []
    for kw in MECHANISM_KEYWORDS.get(drug_code, []):
        if kw.lower() in joined:
            new.append(f'mech:{drug_code}:{kw.lower()}')
    hit_count = len(new)
    if hit_count == 0:
        new.append(f'mech:{drug_code}:no_direct_keyword')
    elif hit_count <= 2:
        new.append(f'mech:{drug_code}:keyword_count_1_2')
    elif hit_count <= 5:
        new.append(f'mech:{drug_code}:keyword_count_3_5')
    else:
        new.append(f'mech:{drug_code}:keyword_count_gt5')
    return sorted(set(tokens + new))


def add_mutation_proxy_tokens(tokens, drug_code):
    tokens = add_mechanism_proxy_tokens(tokens, drug_code)
    if drug_code not in ['CIP', 'NAL']:
        return tokens
    joined = ' '.join([normalize_token_text(t) for t in tokens])
    new = []
    for kw in QUINOLONE_MUTATION_PROXY_TERMS:
        if kw in joined:
            new.append(f'mutation_proxy:{drug_code}:{kw}')
    # Specific combinations often relevant for quinolone resistance interpretation.
    if ('gyra' in joined or 'dna_gyrase' in joined) and ('parc' in joined or 'topoisomerase' in joined):
        new.append(f'mutation_proxy:{drug_code}:gyrase_topoisomerase_context')
    if ('acr' in joined or 'efflux' in joined) and ('mar' in joined or 'rama' in joined or 'soxs' in joined):
        new.append(f'mutation_proxy:{drug_code}:efflux_regulator_context')
    if len(new) == 0:
        new.append(f'mutation_proxy:{drug_code}:no_quionolone_proxy')
    return sorted(set(tokens + new))


def load_optional_amr_tokens(path):
    """Load AMRFinderPlus/CARD/ResFinder-like TSV and convert to genome-level tokens.
    Expected flexible columns: genome_id plus gene/symbol/class/subclass/mechanism/product.
    """
    path = Path(path)
    if not path.exists():
        print('Optional AMR table not found:', path)
        return {}
    df = pd.read_csv(path, sep='\t', low_memory=False)
    cols_norm = {c: norm_col(c) for c in df.columns}
    gid_col = None
    for c, nc in cols_norm.items():
        if 'genome' in nc and 'id' in nc:
            gid_col = c
            break
    if gid_col is None:
        raise ValueError('Optional AMR table cần cột genome_id.')
    token_cols = []
    for c, nc in cols_norm.items():
        if any(k in nc for k in ['gene', 'symbol', 'class', 'subclass', 'mechanism', 'product', 'element']):
            token_cols.append(c)
    token_map = defaultdict(list)
    for _, row in df.iterrows():
        gid = str(row[gid_col])
        for c in token_cols:
            tok = clean_feature_token(row[c])
            if tok:
                token_map[gid].append(f'amr:{norm_col(c)}:{tok}')
    return {gid: sorted(set(toks)) for gid, toks in token_map.items()}

amr_token_map = {}
if USE_OPTIONAL_AMR_TABLE:
    amr_token_map = load_optional_amr_tokens(OPTIONAL_AMR_TABLE_PATH)
    print('Loaded optional AMR token genomes:', len(amr_token_map))


## 5. Optional: sequence-based mutation signatures

Cell này **không bắt buộc**. Nếu bạn có FASTA protein/CDS của các gene `gyrA`, `gyrB`, `parC`, `parE`, có thể đặt vào folder:

```text
/content/salmonella_direction_X_limitation_driven/optional_mutation_fastas/
```

Tên file ví dụ:

```text
gyrA.faa
parC.faa
```

Header FASTA nên chứa `genome_id`. Notebook sẽ tạo token như `mut:gyrA:pos83:F`. Nếu chưa có FASTA, phần này tự skip.


In [ ]:
# =========================
# 6. Optional local FASTA mutation feature extraction
# =========================
def parse_fasta(path):
    records = {}
    header = None
    seq_parts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if header is not None:
                    records[header] = ''.join(seq_parts).upper()
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)
    if header is not None:
        records[header] = ''.join(seq_parts).upper()
    return records


def extract_genome_id_from_header(header):
    # Try common BV-BRC genome id pattern like 12345.678
    m = re.search(r'(\d+\.\d+)', header)
    if m:
        return m.group(1)
    return header.split()[0].split('|')[0]


def load_local_mutation_tokens(fasta_dir):
    fasta_dir = Path(fasta_dir)
    if not fasta_dir.exists():
        print('No optional mutation FASTA dir:', fasta_dir)
        return {}
    mutation_token_map = defaultdict(list)
    gene_files = []
    for gene in ['gyrA', 'gyrB', 'parC', 'parE']:
        for ext in ['faa', 'fa', 'fasta', 'fna']:
            p = fasta_dir / f'{gene}.{ext}'
            if p.exists():
                gene_files.append((gene, p))
    if not gene_files:
        print('No gyrA/gyrB/parC/parE fasta files found in:', fasta_dir)
        return {}
    for gene, p in gene_files:
        recs = parse_fasta(p)
        # Build simple majority consensus by aligned sequence length. This assumes sequences are aligned or same-length proteins.
        seq_by_gid = {extract_genome_id_from_header(h): s for h, s in recs.items() if len(s) > 0}
        if len(seq_by_gid) < 20:
            print('Skip small FASTA:', p.name, len(seq_by_gid))
            continue
        lengths = Counter(len(s) for s in seq_by_gid.values())
        common_len = lengths.most_common(1)[0][0]
        seq_by_gid = {gid: s for gid, s in seq_by_gid.items() if len(s) == common_len}
        arr = np.array([list(s) for s in seq_by_gid.values()])
        consensus = []
        for j in range(arr.shape[1]):
            vals, counts = np.unique(arr[:, j], return_counts=True)
            consensus.append(vals[np.argmax(counts)])
        consensus = ''.join(consensus)
        for gid, seq in seq_by_gid.items():
            for j, aa in enumerate(seq):
                if aa != consensus[j] and aa not in ['X', '-', '*']:
                    mutation_token_map[gid].append(f'mut:{gene}:pos{j+1}:{consensus[j]}>{aa}')
        print('Loaded mutation tokens from', p.name, 'genomes:', len(seq_by_gid))
    return {gid: sorted(set(toks)) for gid, toks in mutation_token_map.items()}

mutation_token_map = {}
if USE_OPTIONAL_LOCAL_MUTATION_FASTA:
    mutation_token_map = load_local_mutation_tokens(OPTIONAL_MUTATION_FASTA_DIR)
    print('Genomes with optional mutation tokens:', len(mutation_token_map))


## 6. Sparse matrix + model utilities


In [ ]:
# =========================
# 7. Sparse matrix and model helpers
# =========================
def build_sparse_matrix(token_lists, min_df=1):
    mlb = MultiLabelBinarizer(sparse_output=True)
    X = mlb.fit_transform(token_lists).astype(np.float32)
    names = np.array(mlb.classes_, dtype=object)
    if min_df > 1 and X.shape[1] > 0:
        dfreq = np.asarray((X > 0).sum(axis=0)).ravel()
        keep = dfreq >= min_df
        X = X[:, keep]
        names = names[keep]
    return X.tocsr(), names


def select_k_chi2(X_train, y_train, X_val, X_test, feature_names, k):
    k = min(int(k), X_train.shape[1])
    if k >= X_train.shape[1]:
        return X_train, X_val, X_test, feature_names, np.arange(X_train.shape[1])
    scores, _ = chi2(X_train, y_train)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    idx = np.argsort(scores)[::-1][:k]
    idx = np.sort(idx)
    return X_train[:, idx], X_val[:, idx], X_test[:, idx], feature_names[idx], idx


def make_model(model_name, y_train=None, random_state=42):
    if model_name in ['lr_balanced', 'dropout_lr']:
        return LogisticRegression(
            max_iter=3000,
            solver='liblinear',
            class_weight='balanced',
            random_state=random_state,
        )
    if model_name == 'rf_balanced':
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=random_state,
        )
    if model_name in ['xgb_weighted', 'dropout_xgb'] and HAS_XGB:
        pos = max(1, int(np.sum(y_train == 1))) if y_train is not None else 1
        neg = max(1, int(np.sum(y_train == 0))) if y_train is not None else 1
        return xgb.XGBClassifier(
            n_estimators=250,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            objective='binary:logistic',
            eval_metric='logloss',
            tree_method='hist',
            scale_pos_weight=neg / pos,
            random_state=random_state,
            n_jobs=-1,
        )
    raise ValueError(f'Unknown model: {model_name}')


def predict_proba_safe(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        z = model.decision_function(X)
        return 1 / (1 + np.exp(-z))
    return model.predict(X)


def best_f1_threshold(y_true, prob):
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        pred = (prob >= t).astype(int)
        f = f1_score(y_true, pred, zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), float(f)
    return best_t, best_f


def metric_dict(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    out = {
        'accuracy': accuracy_score(y_true, pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'auprc': average_precision_score(y_true, prob),
    }
    try:
        out['auroc'] = roc_auc_score(y_true, prob)
    except Exception:
        out['auroc'] = np.nan
    return out


def dropout_feature_masks(n_features, n_models, dropout_rate, random_state):
    rng = np.random.default_rng(random_state)
    masks = []
    keep_n = max(2, int(round(n_features * (1.0 - dropout_rate))))
    for _ in range(n_models):
        idx = np.sort(rng.choice(n_features, size=keep_n, replace=False))
        masks.append(idx)
    return masks


def train_predict_dropout_ensemble(model_base_name, X_train, y_train, X_val, X_test, random_state=42):
    base = 'lr_balanced' if model_base_name == 'dropout_lr' else 'xgb_weighted'
    masks = dropout_feature_masks(X_train.shape[1], DROPOUT_ENSEMBLE_SIZE, DROPOUT_RATE, random_state)
    val_probs = []
    test_probs = []
    models = []
    for j, idx in enumerate(masks):
        m = make_model(base, y_train=y_train, random_state=random_state + j)
        m.fit(X_train[:, idx], y_train)
        val_probs.append(predict_proba_safe(m, X_val[:, idx]))
        test_probs.append(predict_proba_safe(m, X_test[:, idx]))
        models.append((m, idx))
    return np.mean(val_probs, axis=0), np.mean(test_probs, axis=0), models


def top_features_from_model(model, feature_names, top_n=30):
    if hasattr(model, 'coef_'):
        imp = np.abs(model.coef_[0])
    elif hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
    else:
        return pd.DataFrame()
    idx = np.argsort(imp)[::-1][:min(top_n, len(imp))]
    return pd.DataFrame({'feature': feature_names[idx], 'importance': imp[idx]})


## 7. Bias-aware split audit

Notebook tự kiểm tra split nào hợp lệ. Một split theo group chỉ chạy nếu:

- group có đủ số nhóm;
- train/test đều có cả S và R;
- test không quá nhỏ/quá lớn.

Điều này giúp tránh viết quá mạnh khi metadata không đủ.


In [ ]:
# =========================
# 8. Split helpers and bias audit
# =========================
def clean_group_values(s):
    return s.astype(str).replace({'nan': '', 'None': '', 'none': '', '': 'unknown'}).fillna('unknown')


def valid_group_split(model_df, y, col, random_state=42):
    if col not in model_df.columns:
        return None, {'group_column': col, 'available': False, 'reason': 'missing_column'}
    groups = clean_group_values(model_df[col])
    n_groups = groups.nunique()
    largest_frac = groups.value_counts(normalize=True).iloc[0] if n_groups else 1.0
    audit = {
        'group_column': col,
        'available': True,
        'n_groups': int(n_groups),
        'largest_group_frac': float(largest_frac),
        'reason': 'candidate',
    }
    if n_groups < 5:
        audit['available'] = False
        audit['reason'] = 'too_few_groups'
        return None, audit
    if largest_frac > 0.85:
        audit['available'] = False
        audit['reason'] = 'one_group_dominates'
        return None, audit
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=random_state)
    try:
        tr_idx, te_idx = next(gss.split(np.zeros(len(y)), y, groups=groups))
    except Exception as e:
        audit['available'] = False
        audit['reason'] = 'groupshuffle_failed:' + repr(e)[:80]
        return None, audit
    if len(np.unique(y[tr_idx])) < 2 or len(np.unique(y[te_idx])) < 2:
        audit['available'] = False
        audit['reason'] = 'single_class_after_split'
        return None, audit
    frac_test = len(te_idx) / len(y)
    if frac_test < 0.10 or frac_test > 0.40:
        audit['available'] = False
        audit['reason'] = 'bad_test_fraction'
        audit['test_fraction'] = float(frac_test)
        return None, audit
    audit['reason'] = 'ok'
    audit['test_fraction'] = float(frac_test)
    return (tr_idx, te_idx), audit


def temporal_split_if_valid(model_df, y):
    col = 'collection_year'
    audit = {'group_column': col, 'available': False, 'reason': 'missing_or_invalid'}
    if col not in model_df.columns:
        return None, audit
    years = pd.to_numeric(model_df[col], errors='coerce')
    valid = years.notna()
    if valid.mean() < 0.7:
        audit['reason'] = 'too_many_missing_years'
        audit['valid_year_fraction'] = float(valid.mean())
        return None, audit
    q = years.quantile(0.80)
    tr_idx = np.where(years < q)[0]
    te_idx = np.where(years >= q)[0]
    if len(tr_idx) < 50 or len(te_idx) < 30:
        audit['reason'] = 'too_small_after_temporal_split'
        return None, audit
    if len(np.unique(y[tr_idx])) < 2 or len(np.unique(y[te_idx])) < 2:
        audit['reason'] = 'single_class_after_temporal_split'
        return None, audit
    audit.update({'available': True, 'reason': 'ok', 'threshold_year': float(q), 'test_fraction': float(len(te_idx)/len(y))})
    return (tr_idx, te_idx), audit


def get_split_jobs(model_df, y, repeat_seed):
    # Always include stratified random.
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=TEST_SIZE,
        stratify=y,
        random_state=repeat_seed,
    )
    jobs = [('stratified_random', train_idx, test_idx, {'group_column': 'none', 'available': True, 'reason': 'ok'})]

    audits = []
    if RUN_BIAS_AWARE_SPLITS:
        # Temporal split only once per repeat, deterministic.
        split, audit = temporal_split_if_valid(model_df, y)
        audits.append(audit)
        if split is not None:
            jobs.append(('temporal_latest_years', split[0], split[1], audit))

        for col in BIAS_GROUP_COLUMNS:
            if col == 'collection_year':
                continue
            split, audit = valid_group_split(model_df, y, col, random_state=repeat_seed)
            audits.append(audit)
            if split is not None:
                jobs.append((f'group_{col}', split[0], split[1], audit))
            if len(jobs) >= MAX_SPLITS_PER_DRUG:
                break
    return jobs, audits


## 8. Main evaluation function

Điểm khác Direction V:

- thêm `dropout_lr` / `dropout_xgb` ensemble để giảm phụ thuộc một marker;
- thêm feature set mutation-proxy cho CIP/NAL;
- thêm bias-aware split;
- thêm negative control.


In [ ]:
# =========================
# 9. Evaluation core
# =========================
def train_eval_one_split(X, y, feature_names, train_idx, test_idx, model_name, k_features, repeat_seed):
    # Inner train/val split for threshold tuning.
    y_train_full = y[train_idx]
    tr_inner, val_inner = train_test_split(
        train_idx,
        test_size=VAL_SIZE,
        stratify=y_train_full,
        random_state=repeat_seed,
    )
    X_tr, y_tr = X[tr_inner], y[tr_inner]
    X_val, y_val = X[val_inner], y[val_inner]
    X_te, y_te = X[test_idx], y[test_idx]

    X_tr_k, X_val_k, X_te_k, names_k, selected_idx = select_k_chi2(X_tr, y_tr, X_val, X_te, feature_names, k_features)

    if model_name in ['dropout_lr', 'dropout_xgb']:
        val_prob, test_prob, models = train_predict_dropout_ensemble(
            model_name, X_tr_k, y_tr, X_val_k, X_te_k, random_state=repeat_seed
        )
        # For top features, train one full model on selected features.
        ref_model_name = 'lr_balanced' if model_name == 'dropout_lr' else 'xgb_weighted'
        ref_model = make_model(ref_model_name, y_train=y_tr, random_state=repeat_seed)
        ref_model.fit(X_tr_k, y_tr)
        model_for_top = ref_model
    else:
        model = make_model(model_name, y_train=y_tr, random_state=repeat_seed)
        model.fit(X_tr_k, y_tr)
        val_prob = predict_proba_safe(model, X_val_k)
        test_prob = predict_proba_safe(model, X_te_k)
        model_for_top = model

    threshold, val_f1 = best_f1_threshold(y_val, val_prob)
    m = metric_dict(y_te, test_prob, threshold)
    m.update({
        'threshold': threshold,
        'val_f1_best': val_f1,
        'n_train': int(len(tr_inner)),
        'n_val': int(len(val_inner)),
        'n_test': int(len(test_idx)),
        'n_selected_features': int(len(names_k)),
    })
    top_df = top_features_from_model(model_for_top, names_k, top_n=50)
    return m, top_df


def evaluate_feature_set(drug_code, model_df, X, y, feature_names, feature_set_name):
    metric_rows = []
    top_rows = []
    audit_rows = []
    for rep in range(N_REPEATS):
        repeat_seed = RANDOM_SEED + rep * 17
        split_jobs, audits = get_split_jobs(model_df, y, repeat_seed)
        for a in audits:
            row = dict(a)
            row.update({'drug_code': drug_code, 'repeat': rep})
            audit_rows.append(row)
        for split_mode, train_idx, test_idx, split_audit in split_jobs:
            for k in K_FEATURES_LIST:
                for model_name in MODEL_NAMES:
                    # Skip XGB dropout if xgboost unavailable.
                    if model_name in ['xgb_weighted', 'dropout_xgb'] and not HAS_XGB:
                        continue
                    try:
                        m, top_df = train_eval_one_split(
                            X, y, feature_names, train_idx, test_idx,
                            model_name=model_name,
                            k_features=k,
                            repeat_seed=repeat_seed,
                        )
                        row = {
                            'drug_code': drug_code,
                            'feature_set': feature_set_name,
                            'split_mode': split_mode,
                            'model': model_name,
                            'k_features': k,
                            'repeat': rep,
                        }
                        row.update(m)
                        metric_rows.append(row)
                        if not top_df.empty:
                            top_df = top_df.copy()
                            top_df['drug_code'] = drug_code
                            top_df['feature_set'] = feature_set_name
                            top_df['split_mode'] = split_mode
                            top_df['model'] = model_name
                            top_df['k_features'] = k
                            top_df['repeat'] = rep
                            top_rows.append(top_df)
                    except Exception as e:
                        print('Eval failed:', drug_code, feature_set_name, split_mode, model_name, k, repr(e)[:120])
    return pd.DataFrame(metric_rows), pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame(), pd.DataFrame(audit_rows)


def negative_control_eval(drug_code, model_df, token_lists, feature_set_name):
    if not RUN_NEGATIVE_CONTROL:
        return pd.DataFrame()
    X, feature_names = build_sparse_matrix(token_lists, min_df=1)
    y_true = model_df['y'].astype(int).values
    rng = np.random.default_rng(RANDOM_SEED)
    y = rng.permutation(y_true)
    rows = []
    # keep it light: LR + k=200, stratified random only
    for rep in range(min(3, N_REPEATS)):
        seed = RANDOM_SEED + 1000 + rep
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=seed)
        for model_name in ['lr_balanced']:
            k = min(200, X.shape[1])
            try:
                m, _ = train_eval_one_split(X, y, feature_names, train_idx, test_idx, model_name, k, seed)
                row = {'drug_code': drug_code, 'feature_set': feature_set_name, 'model': model_name, 'k_features': k, 'repeat': rep}
                row.update(m)
                rows.append(row)
            except Exception as e:
                print('Negative control failed:', drug_code, repr(e)[:120])
    return pd.DataFrame(rows)


## 9. Feature-dropout stress test

Stress test này trả lời câu hỏi:

> Nếu bỏ top 1 / top 5 / top 10 features rồi retrain, performance còn giữ được không?

Nếu giảm ít, model không phụ thuộc quá mạnh vào một marker. Nếu giảm nhiều, cần viết là model còn dễ over-rely.


In [ ]:
# =========================
# 10. Top-feature removal stress test
# =========================
def feature_dropout_stress_test(drug_code, model_df, X, y, feature_names, feature_set_name, best_model='dropout_lr', best_k=500):
    if not RUN_FEATURE_DROPOUT_STRESS_TEST:
        return pd.DataFrame()
    rows = []
    for rep in range(min(5, N_REPEATS)):
        seed = RANDOM_SEED + 3000 + rep
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=seed)
        y_train = y[train_idx]
        tr_inner, val_inner = train_test_split(train_idx, test_size=VAL_SIZE, stratify=y_train, random_state=seed)
        X_tr, y_tr = X[tr_inner], y[tr_inner]
        X_val, y_val = X[val_inner], y[val_inner]
        X_te, y_te = X[test_idx], y[test_idx]

        # Select initial top-k using train only.
        X_tr_k, X_val_k, X_te_k, names_k, selected_idx = select_k_chi2(X_tr, y_tr, X_val, X_te, feature_names, best_k)
        scores, _ = chi2(X_tr_k, y_tr)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
        rank = np.argsort(scores)[::-1]

        conditions = [('no_removal', 0), ('remove_top1', 1), ('remove_top5', 5), ('remove_top10', 10)]
        rng = np.random.default_rng(seed)
        random_remove = min(5, X_tr_k.shape[1] // 4)
        random_idx = rng.choice(X_tr_k.shape[1], size=random_remove, replace=False) if random_remove > 0 else np.array([], dtype=int)

        for cond, n_remove in conditions:
            remove_idx = rank[:n_remove] if n_remove > 0 else np.array([], dtype=int)
            keep = np.ones(X_tr_k.shape[1], dtype=bool)
            keep[remove_idx] = False
            if keep.sum() < 2:
                continue
            Xtr2, Xval2, Xte2 = X_tr_k[:, keep], X_val_k[:, keep], X_te_k[:, keep]
            names2 = names_k[keep]
            model_name = best_model
            if model_name in ['dropout_lr', 'dropout_xgb']:
                val_prob, test_prob, _ = train_predict_dropout_ensemble(model_name, Xtr2, y_tr, Xval2, Xte2, random_state=seed)
            else:
                model = make_model(model_name, y_train=y_tr, random_state=seed)
                model.fit(Xtr2, y_tr)
                val_prob = predict_proba_safe(model, Xval2)
                test_prob = predict_proba_safe(model, Xte2)
            thr, _ = best_f1_threshold(y_val, val_prob)
            m = metric_dict(y_te, test_prob, thr)
            rows.append({
                'drug_code': drug_code,
                'feature_set': feature_set_name,
                'condition': cond,
                'repeat': rep,
                'model': model_name,
                'k_features_initial': int(best_k),
                'n_removed': int(n_remove),
                'removed_features': ';'.join(map(str, names_k[remove_idx][:20])),
                **m,
            })

        if len(random_idx) > 0:
            keep = np.ones(X_tr_k.shape[1], dtype=bool)
            keep[random_idx] = False
            Xtr2, Xval2, Xte2 = X_tr_k[:, keep], X_val_k[:, keep], X_te_k[:, keep]
            model_name = best_model
            if model_name in ['dropout_lr', 'dropout_xgb']:
                val_prob, test_prob, _ = train_predict_dropout_ensemble(model_name, Xtr2, y_tr, Xval2, Xte2, random_state=seed)
            else:
                model = make_model(model_name, y_train=y_tr, random_state=seed)
                model.fit(Xtr2, y_tr)
                val_prob = predict_proba_safe(model, Xval2)
                test_prob = predict_proba_safe(model, Xte2)
            thr, _ = best_f1_threshold(y_val, val_prob)
            m = metric_dict(y_te, test_prob, thr)
            rows.append({
                'drug_code': drug_code,
                'feature_set': feature_set_name,
                'condition': 'remove_random5',
                'repeat': rep,
                'model': model_name,
                'k_features_initial': int(best_k),
                'n_removed': int(len(random_idx)),
                'removed_features': ';'.join(map(str, names_k[random_idx][:20])),
                **m,
            })
    return pd.DataFrame(rows)


## 10. Run Direction X

Nếu muốn chạy nhanh trước:

```python
TARGET_DRUGS = TARGET_DRUGS[:2]   # TET, CIP
N_REPEATS = 3
K_FEATURES_LIST = [200, 500]
```

Nếu muốn tập trung tăng NAL/CIP:

```python
TARGET_DRUGS = [d for d in TARGET_DRUGS if d['drug_code'] in ['CIP', 'NAL']]
N_REPEATS = 10
```


In [ ]:
# =========================
# 11. Run all drugs
# =========================
all_metrics = []
all_tops = []
all_audits = []
all_neg = []
all_stress = []
all_dataset_rows = []

for drug_info in TARGET_DRUGS:
    drug_code = drug_info['drug_code']
    print('\n' + '=' * 100)
    print('Direction X drug:', drug_code)
    print('=' * 100)
    try:
        model_df, tokens_by_genome, data_source = load_or_build_drug_dataset(drug_info)
    except Exception as e:
        print('SKIP drug due to data error:', drug_code, repr(e)[:300])
        continue

    if 'y' not in model_df.columns:
        model_df['y'] = (model_df['phenotype'] == 'R').astype(int)
    model_df['genome_id'] = model_df['genome_id'].astype(str)
    model_df = model_df.dropna(subset=['y']).copy().reset_index(drop=True)

    base_token_lists = []
    valid_idx = []
    for i, row in model_df.iterrows():
        gid = str(row['genome_id'])
        toks = tokens_by_genome.get(gid, [])
        if isinstance(toks, list) and len(toks) > 0:
            base_token_lists.append(toks)
            valid_idx.append(i)
    model_df = model_df.loc[valid_idx].reset_index(drop=True)
    y = model_df['y'].astype(int).values

    if len(model_df) < 80 or len(np.unique(y)) < 2:
        print('SKIP: usable rows too few or single class.', model_df.shape, np.unique(y, return_counts=True))
        continue

    # Build feature sets.
    mechanism_lists = [add_mechanism_proxy_tokens(toks, drug_code) for toks in base_token_lists]
    mutation_proxy_lists = [add_mutation_proxy_tokens(toks, drug_code) for toks in base_token_lists]

    optional_amr_lists = []
    if USE_OPTIONAL_AMR_TABLE and amr_token_map:
        for toks, gid in zip(mutation_proxy_lists, model_df['genome_id'].astype(str)):
            optional_amr_lists.append(sorted(set(toks + amr_token_map.get(gid, []))))

    optional_mut_lists = []
    if USE_OPTIONAL_LOCAL_MUTATION_FASTA and mutation_token_map:
        for toks, gid in zip(mutation_proxy_lists, model_df['genome_id'].astype(str)):
            optional_mut_lists.append(sorted(set(toks + mutation_token_map.get(gid, []))))

    feature_sets = {
        'base_annotation': base_token_lists,
        'mechanism_proxy': mechanism_lists,
        'mutation_proxy': mutation_proxy_lists,
    }
    if optional_amr_lists:
        feature_sets['mechanism_plus_optional_amr'] = optional_amr_lists
    if optional_mut_lists:
        feature_sets['true_mutation_signature'] = optional_mut_lists

    all_dataset_rows.append({
        'drug_code': drug_code,
        'data_source': data_source,
        'n_rows': int(len(model_df)),
        'n_S': int(np.sum(y == 0)),
        'n_R': int(np.sum(y == 1)),
        'feature_sets': ','.join(feature_sets.keys()),
    })
    model_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_model_rows_used.csv', index=False)

    for feature_set_name, token_lists in feature_sets.items():
        print('\nFeature set:', drug_code, feature_set_name)
        X, feature_names = build_sparse_matrix(token_lists, min_df=1)
        print('X shape:', X.shape)
        metrics_df, top_df, audit_df = evaluate_feature_set(drug_code, model_df, X, y, feature_names, feature_set_name)
        if not metrics_df.empty:
            metrics_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_{feature_set_name}_metrics.csv', index=False)
            all_metrics.append(metrics_df)
        if not top_df.empty:
            top_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_{feature_set_name}_top_features_raw.csv', index=False)
            all_tops.append(top_df)
        if not audit_df.empty:
            all_audits.append(audit_df)

        # Negative control only for strongest/new feature set to save time.
        if feature_set_name in ['mutation_proxy', 'true_mutation_signature']:
            neg_df = negative_control_eval(drug_code, model_df, token_lists, feature_set_name)
            if not neg_df.empty:
                neg_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_{feature_set_name}_negative_control.csv', index=False)
                all_neg.append(neg_df)

    # Stress test on mutation_proxy because this is the main Direction X feature set.
    X_main, names_main = build_sparse_matrix(mutation_proxy_lists, min_df=1)
    stress_df = feature_dropout_stress_test(
        drug_code, model_df, X_main, y, names_main,
        feature_set_name='mutation_proxy',
        best_model='dropout_xgb' if HAS_XGB else 'dropout_lr',
        best_k=500,
    )
    if not stress_df.empty:
        stress_df.to_csv(OUTPUT_DIR / f'direction_X_{drug_code}_feature_dropout_stress_test.csv', index=False)
        all_stress.append(stress_df)

print('Done Direction X main loop.')


## 11. Aggregate + annotate top features


In [ ]:
# =========================
# 12. Aggregate results
# =========================
metrics_all = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
top_all = pd.concat(all_tops, ignore_index=True) if all_tops else pd.DataFrame()
audit_all = pd.concat(all_audits, ignore_index=True) if all_audits else pd.DataFrame()
neg_all = pd.concat(all_neg, ignore_index=True) if all_neg else pd.DataFrame()
stress_all = pd.concat(all_stress, ignore_index=True) if all_stress else pd.DataFrame()
dataset_summary = pd.DataFrame(all_dataset_rows)

if not dataset_summary.empty:
    dataset_summary.to_csv(OUTPUT_DIR / 'direction_X_dataset_summary.csv', index=False)
if not metrics_all.empty:
    metrics_all.to_csv(OUTPUT_DIR / 'direction_X_all_metrics.csv', index=False)
if not top_all.empty:
    top_all.to_csv(OUTPUT_DIR / 'direction_X_all_top_features_raw.csv', index=False)
if not audit_all.empty:
    audit_all.to_csv(OUTPUT_DIR / 'direction_X_bias_split_audit.csv', index=False)
if not neg_all.empty:
    neg_all.to_csv(OUTPUT_DIR / 'direction_X_negative_control_all.csv', index=False)
if not stress_all.empty:
    stress_all.to_csv(OUTPUT_DIR / 'direction_X_feature_dropout_stress_test.csv', index=False)

print('Dataset summary:')
display(dataset_summary)

if metrics_all.empty:
    raise RuntimeError('No metrics. Check data/fetch/errors above.')

summary = (
    metrics_all
    .groupby(['drug_code', 'feature_set', 'split_mode', 'model', 'k_features'])
    .agg(
        n_runs=('f1', 'size'),
        f1_mean=('f1', 'mean'),
        f1_std=('f1', 'std'),
        bal_acc_mean=('balanced_accuracy', 'mean'),
        bal_acc_std=('balanced_accuracy', 'std'),
        auprc_mean=('auprc', 'mean'),
        auprc_std=('auprc', 'std'),
        auroc_mean=('auroc', 'mean'),
        precision_mean=('precision', 'mean'),
        recall_mean=('recall', 'mean'),
        threshold_mean=('threshold', 'mean'),
    )
    .reset_index()
)
summary.to_csv(OUTPUT_DIR / 'direction_X_metric_summary_all_settings.csv', index=False)

best_summary = (
    summary
    .sort_values(['drug_code', 'split_mode', 'f1_mean', 'bal_acc_mean', 'auprc_mean'], ascending=[True, True, False, False, False])
    .groupby(['drug_code', 'split_mode'], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_summary.to_csv(OUTPUT_DIR / 'direction_X_best_summary.csv', index=False)

print('Best summary:')
display(best_summary)

# Compare base vs Direction X feature sets on stratified random.
compare = summary[summary['split_mode'] == 'stratified_random'].copy()
if not compare.empty:
    best_by_feature = (
        compare.sort_values(['drug_code', 'feature_set', 'f1_mean'], ascending=[True, True, False])
        .groupby(['drug_code', 'feature_set'], as_index=False)
        .head(1)
    )
    best_by_feature.to_csv(OUTPUT_DIR / 'direction_X_best_by_feature_set_stratified.csv', index=False)
    print('Best by feature set, stratified random:')
    display(best_by_feature)

if not audit_all.empty:
    audit_compact = audit_all.drop_duplicates(['drug_code', 'group_column', 'reason']).sort_values(['drug_code', 'group_column'])
    print('Bias split audit compact:')
    display(audit_compact.head(40))

if not neg_all.empty:
    neg_summary = (
        neg_all.groupby(['drug_code', 'feature_set', 'model', 'k_features'])
        .agg(f1_mean=('f1', 'mean'), bal_acc_mean=('balanced_accuracy', 'mean'), auprc_mean=('auprc', 'mean'))
        .reset_index()
    )
    neg_summary.to_csv(OUTPUT_DIR / 'direction_X_negative_control_summary.csv', index=False)
    print('Negative control summary:')
    display(neg_summary)

if not stress_all.empty:
    stress_summary = (
        stress_all.groupby(['drug_code', 'feature_set', 'condition'])
        .agg(
            f1_mean=('f1', 'mean'),
            f1_std=('f1', 'std'),
            bal_acc_mean=('balanced_accuracy', 'mean'),
            auprc_mean=('auprc', 'mean'),
        )
        .reset_index()
    )
    stress_summary.to_csv(OUTPUT_DIR / 'direction_X_feature_dropout_stress_summary.csv', index=False)
    print('Feature dropout stress summary:')
    display(stress_summary)


In [ ]:
# =========================
# 13. Top feature interpretation
# =========================
INTERPRET_KEYWORDS = {
    'tetracycline': ['tet', 'tetracycline'],
    'aminoglycoside': ['aac', 'aph', 'ant', 'aad', 'aminoglycoside', 'acetyltransferase', 'phosphotransferase', 'nucleotidyltransferase'],
    'quinolone_mutation_proxy': ['gyr', 'gyra', 'gyrb', 'parc', 'pare', 'topoisomerase', 'qnr'],
    'efflux_regulation': ['efflux', 'acr', 'mar', 'marr', 'rama', 'soxs', 'oqx', 'qep'],
    'mobile_element': ['integrase', 'integron', 'inti1', 'transposase', 'plasmid'],
    'mechanism_proxy': ['mech:', 'mutation_proxy:', 'mut:'],
}


def annotate_feature(feature):
    s = str(feature).lower()
    hits = []
    for label, kws in INTERPRET_KEYWORDS.items():
        if any(kw in s for kw in kws):
            hits.append(label)
    if not hits:
        if s.startswith('pgfam') or s.startswith('plfam'):
            hits.append('protein_family_unknown')
        else:
            hits.append('other_annotation')
    return ';'.join(hits)


def summarize_top_features(top_all, top_n=50):
    if top_all.empty:
        return pd.DataFrame()
    df = top_all.copy()
    df['annotation_group'] = df['feature'].map(annotate_feature)
    # Count appearances and average importance.
    out = (
        df.groupby(['drug_code', 'feature_set', 'split_mode', 'feature', 'annotation_group'])
        .agg(
            n_appear=('feature', 'size'),
            mean_importance=('importance', 'mean'),
            max_importance=('importance', 'max'),
        )
        .reset_index()
        .sort_values(['drug_code', 'feature_set', 'split_mode', 'n_appear', 'mean_importance'], ascending=[True, True, True, False, False])
    )
    return out.groupby(['drug_code', 'feature_set', 'split_mode'], as_index=False).head(top_n).reset_index(drop=True)

feature_summary = summarize_top_features(top_all, top_n=50)
if not feature_summary.empty:
    feature_summary.to_csv(OUTPUT_DIR / 'direction_X_top_feature_summary_annotated.csv', index=False)
    print('Annotated top features:')
    display(feature_summary.head(40))

# Drug-level annotation hit count.
if not feature_summary.empty:
    interpretation_summary = (
        feature_summary
        .assign(is_direct=lambda d: d['annotation_group'].str.contains('tetracycline|aminoglycoside|quinolone|efflux|mobile|mechanism', regex=True))
        .groupby(['drug_code', 'feature_set', 'split_mode'])
        .agg(
            top_features=('feature', 'count'),
            direct_or_mechanism_hits=('is_direct', 'sum'),
            example_hits=('feature', lambda x: '; '.join(list(x.head(8))))
        )
        .reset_index()
    )
    interpretation_summary.to_csv(OUTPUT_DIR / 'direction_X_feature_interpretation_summary.csv', index=False)
    print('Feature interpretation summary:')
    display(interpretation_summary)


## 12. Plots


In [ ]:
# =========================
# 14. Plots
# =========================
# 1) Best F1 per drug/split
plot_df = best_summary.copy()
if not plot_df.empty:
    plot_df['label'] = plot_df['drug_code'] + '\n' + plot_df['split_mode'].astype(str).str.replace('_', ' ')
    plt.figure(figsize=(12, 5))
    plt.bar(plot_df['label'], plot_df['f1_mean'], yerr=plot_df['f1_std'].fillna(0), capsize=4)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Mean F1')
    plt.title('Direction X: best robust performance by drug and split')
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_direction_X_best_f1_by_drug_split.png', dpi=200)
    plt.show()

# 2) Feature set comparison on stratified random.
if 'best_by_feature' in globals() and not best_by_feature.empty:
    b = best_by_feature.copy()
    b['label'] = b['drug_code'] + '\n' + b['feature_set'].astype(str).str.replace('_', ' ')
    plt.figure(figsize=(13, 5))
    plt.bar(b['label'], b['f1_mean'], yerr=b['f1_std'].fillna(0), capsize=4)
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Best mean F1')
    plt.title('Direction X: base vs mechanism/mutation-aware feature sets')
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_direction_X_feature_set_comparison.png', dpi=200)
    plt.show()

# 3) Feature dropout stress.
if 'stress_summary' in globals() and not stress_summary.empty:
    ss = stress_summary.copy()
    for drug in ss['drug_code'].unique():
        d = ss[ss['drug_code'] == drug].copy()
        plt.figure(figsize=(8, 4))
        plt.bar(d['condition'], d['f1_mean'], yerr=d['f1_std'].fillna(0), capsize=4)
        plt.xticks(rotation=30, ha='right')
        plt.ylabel('Mean F1')
        plt.title(f'Direction X stress test: {drug}')
        plt.ylim(0, 1.05)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'fig_direction_X_feature_dropout_stress_{drug}.png', dpi=200)
        plt.show()


## 13. Auto conclusion

Cell cuối tạo đoạn kết luận tự động để đưa vào báo cáo.


In [ ]:
# =========================
# 15. Auto conclusion + zip outputs
# =========================
def fmt(x, nd=3):
    try:
        if pd.isna(x):
            return 'NA'
        return f'{float(x):.{nd}f}'
    except Exception:
        return str(x)

lines = []
lines.append('# AUTO CONCLUSION — Direction X')
lines.append('')
lines.append('## Goal')
lines.append('Direction X was designed as a limitation-driven extension of the previous external validation pipeline. It targets mutation-driven resistance, metadata/sampling bias, and single-marker over-reliance.')
lines.append('')

lines.append('## Best robust results')
if not best_summary.empty:
    for _, r in best_summary.iterrows():
        lines.append(f"- {r['drug_code']} / {r['split_mode']}: best={r['feature_set']} + {r['model']} k={int(r['k_features'])}, F1={fmt(r['f1_mean'])}±{fmt(r['f1_std'])}, bal.acc={fmt(r['bal_acc_mean'])}, AUPRC={fmt(r['auprc_mean'])}.")
else:
    lines.append('- No best summary available.')
lines.append('')

lines.append('## Limitation addressed')
lines.append('- Annotation-only features were extended with antibiotic-specific mechanism and mutation-proxy features, especially for CIP/NAL quinolone resistance.')
lines.append('- Bias-aware splits were attempted using serovar, country, host, source and collection year when metadata were sufficient.')
lines.append('- Feature-dropout stress tests quantified whether the model depends excessively on a few dominant markers.')
lines.append('- Shuffled-label negative controls were included to detect pipeline artifacts.')
lines.append('')

if not audit_all.empty:
    ok_audit = audit_all[audit_all['reason'] == 'ok']
    lines.append('## Bias-aware validation audit')
    if ok_audit.empty:
        lines.append('- No metadata-based group split was fully valid; report this as a remaining limitation rather than overclaiming bias robustness.')
    else:
        valid_modes = sorted(set(ok_audit['group_column'].astype(str)))
        lines.append('- Valid metadata/group split signals were available for: ' + ', '.join(valid_modes) + '.')
    lines.append('')

if not neg_all.empty:
    neg_summary = pd.read_csv(OUTPUT_DIR / 'direction_X_negative_control_summary.csv') if (OUTPUT_DIR / 'direction_X_negative_control_summary.csv').exists() else pd.DataFrame()
    lines.append('## Negative control')
    if not neg_summary.empty:
        for _, r in neg_summary.iterrows():
            lines.append(f"- {r['drug_code']} shuffled-label control: F1={fmt(r['f1_mean'])}, bal.acc={fmt(r['bal_acc_mean'])}, AUPRC={fmt(r['auprc_mean'])}.")
    lines.append('')

if not stress_all.empty:
    lines.append('## Feature-dropout robustness')
    stress_summary_path = OUTPUT_DIR / 'direction_X_feature_dropout_stress_summary.csv'
    if stress_summary_path.exists():
        ss = pd.read_csv(stress_summary_path)
        for drug in ss['drug_code'].unique():
            d = ss[ss['drug_code'] == drug]
            base = d[d['condition'] == 'no_removal']
            top5 = d[d['condition'] == 'remove_top5']
            if len(base) and len(top5):
                delta = float(top5['f1_mean'].iloc[0] - base['f1_mean'].iloc[0])
                lines.append(f'- {drug}: removing top 5 selected features changed F1 by {delta:+.3f}.')
    lines.append('')

lines.append('## Suggested wording')
lines.append('Direction X should not be described as proving that all limitations are solved. A safer claim is:')
lines.append('')
lines.append('> We address key limitations of annotation-only Salmonella AMR prediction by adding antibiotic-specific mechanism and mutation-proxy features, bias-aware validation, negative controls, and feature-dropout robustness checks. This improves the evidence that the adaptive pipeline is not merely fitting dataset-specific markers, while highlighting that true mutation-level validation for quinolones still requires sequence-derived gyrA/parC/parE features.')
lines.append('')

report = '\n'.join(lines)
report_path = OUTPUT_DIR / 'AUTO_CONCLUSION_DIRECTION_X.md'
report_path.write_text(report, encoding='utf-8')
print(report)

# Zip outputs
zip_base = BASE_DIR / 'direction_X_outputs'
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('Saved report:', report_path)
print('Saved zip:', zip_path)


## 14. Sau khi chạy xong cần xem gì?

Ưu tiên xem 5 file:

```text
direction_X_best_summary.csv
direction_X_best_by_feature_set_stratified.csv
direction_X_bias_split_audit.csv
direction_X_feature_dropout_stress_summary.csv
AUTO_CONCLUSION_DIRECTION_X.md
```

Cách đọc nhanh:

- Nếu `mutation_proxy` thắng `base_annotation`, bạn có bằng chứng rõ cho Direction X.
- Nếu NAL tăng rõ, đây là điểm mới mạnh nhất.
- Nếu feature-dropout giảm ít khi bỏ top 5, có thể viết model ít phụ thuộc single marker hơn.
- Nếu bias-aware split hợp lệ và vẫn giữ F1 tốt, câu chuyện robustness mạnh hơn.
- Nếu metadata split không hợp lệ, viết là “attempted and audited”, không claim quá mức.
